In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

CHECKPOINT_PATH = Path('/content/drive/MyDrive/thesis/final/casia-full/checkpoints/best_checkpoint.pt')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/thesis/final/casia-full/inference')

In [ ]:
import subprocess, sys

REPO_URL = "https://github.com/juhenes/ngiml"
REPO_BRANCH = "FurtherEnhancement"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")


In [ ]:
from tools.infer_helpers import get_model_complexity_stats, run_prepared_dataset_inference

HF_DATASET_REPO_ID = 'juhenes/ngiml-test'
HF_SNAPSHOT_LOCAL_DIR = Path('/content/hf_datasets/ngiml_test')

INFERENCE_STRATEGY = 'direct'
THRESHOLD_FOR_METRICS = None
PLOT_BINARY_THRESHOLD = 0.5
DIRECT_BATCH_SIZE = 64

In [ ]:
run = run_prepared_dataset_inference(
    checkpoint_path=CHECKPOINT_PATH,
    hf_dataset_repo_id=HF_DATASET_REPO_ID,
    output_root=DRIVE_OUTPUT_ROOT,
    hf_snapshot_local_dir=HF_SNAPSHOT_LOCAL_DIR,
    inference_strategy=INFERENCE_STRATEGY,
    threshold_for_metrics=THRESHOLD_FOR_METRICS,
    plot_binary_threshold=PLOT_BINARY_THRESHOLD,
    direct_batch_size=DIRECT_BATCH_SIZE,
)

print('Snapshot:', run['snapshot_path'])
print('Device:', run['device'])
print('Normalization:', run['normalization_mode'])
print('Threshold for CSV metrics:', run['threshold_for_metrics'])
print('Plot threshold:', run['plot_binary_threshold'])
print('Direct batch size:', run['direct_batch_size'])
print('Saved full CSV:', run['results_csv'])
print('Saved summary CSV:', run['summary_csv'])
print('Saved metric comparison CSV:', run['comparison_csv'])
print('Saved plot root:', run['plot_output_dir'])
display(run['summary_df'])
display(run['comparison_df'])

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

from tools.infer_helpers import load_model_from_checkpoint

import json
CHECKPOINTS_CONFIG_PATH = Path("checkpoints/config.json")
if not CHECKPOINTS_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Checkpoints config not found: {CHECKPOINTS_CONFIG_PATH}")
with open(CHECKPOINTS_CONFIG_PATH, "r") as f:
    checkpoints_config = json.load(f)

RUN_OUTPUT_DIR = Path(checkpoints_config.get("output_dir", "output"))
CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints"
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {CHECKPOINT_DIR}")

TARGET_EPOCH = checkpoints_config.get("target_epoch", None)

if TARGET_EPOCH is None:
    best_ckpt = CHECKPOINT_DIR / "best_checkpoint.pt"
    if not best_ckpt.exists():
        raise FileNotFoundError("best_checkpoint.pt not found.")
    checkpoint_paths = [best_ckpt]
    CKPT_PATH = best_ckpt

else:
    target_ckpt = CHECKPOINT_DIR / f"checkpoint_epoch_{TARGET_EPOCH:03d}.pt"

    if target_ckpt.exists():
        checkpoint_paths = [target_ckpt]
        CKPT_PATH = target_ckpt
    else:
        checkpoint_paths = sorted(
            CHECKPOINT_DIR.glob(f"checkpoint_epoch_{TARGET_EPOCH}*.pt"),
            key=lambda p: p.stat().st_mtime
        )

        if not checkpoint_paths:
            raise FileNotFoundError(
                f"No checkpoint found for epoch {TARGET_EPOCH}."
            )
        CKPT_PATH = checkpoint_paths[-1]

model, _, ckpt_info = load_model_from_checkpoint(CKPT_PATH)
model = model.cpu().eval()

input_size = int(ckpt_info.get("input_size", 448))

class _NgimlForProfiling(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        out = self.base_model(x, target_size=x.shape[-2:], residual_noise=None)
        if isinstance(out, (list, tuple)):
            return out[0]
        return out

wrapper = _NgimlForProfiling(model).eval()
dummy = torch.randn(1, 3, input_size, input_size, dtype=torch.float32)

try:
    from thop import clever_format, profile
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop"]),
    from thop import clever_format, profile

with torch.no_grad():
    macs, params = profile(wrapper, inputs=(dummy,), verbose=False)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
flops = 2.0 * macs

macs_hr, params_hr = clever_format([macs, params], "%.3f")
flops_hr = clever_format([flops], "%.3f")

print("Checkpoint:", CKPT_PATH)
print("Target epoch:", TARGET_EPOCH)
print("Input shape:", tuple(dummy.shape))
print("Trainable params:", f"{trainable_params:,}")
print("Total params:", f"{total_params:,}")
print("THOP params:", params_hr)
print("MACs:", macs_hr)
print("Approx FLOPs (2 * MACs):", flops_hr)